# Filter Electrical Conductivity Papers

This notebook filters the pre-existing `thermocatalysis_keywords_and_LLM` split (1333 papers) with new keywords for electrical conductivity.

In [ ]:
from datasets import load_dataset

# Load the pre-filtered split (1333 papers)
dataset = load_dataset(
    "LeMaterial/LeMat-Synth-Papers",
    "full",
    split="thermocatalysis_keywords_and_LLM",
    token=True,
)

print(f"Loaded {len(dataset)} papers")
print(f"Columns: {dataset.column_names}")

In [ ]:
# Preview the data
dataset.to_pandas()[["id", "title", "pdf_url"]].head()

In [ ]:
def filter_dataset(ds, text_column, list_keywords, exclude_keywords=None):
    """Filter dataset by keywords in a text column."""

    def matches_any_keyword(example):
        if example[text_column] is None:
            return False
        text_lower = example[text_column].lower()
        return any(kw.lower() in text_lower for kw in list_keywords)

    filtered = ds.filter(matches_any_keyword)

    if exclude_keywords:

        def matches_exclude(example):
            if example[text_column] is None:
                return False
            text_lower = example[text_column].lower()
            return any(ek.lower() in text_lower for ek in exclude_keywords)

        filtered = filtered.filter(lambda x: not matches_exclude(x))

    return filtered

In [ ]:
# Define keywords for electrical conductivity
text_column = "abstract"

list_keywords = [
    # Core conductivity terms
    "electrical conductivity",
    "electronic conductivity",
    # "ionic conductivity",
    # "electrical resistivity",
    # "sheet resistance",
    # "conductance",
    # Measurement-related
    # "four-point probe",
    # "four point probe",
    # "van der Pauw",
    # "impedance spectroscopy",
    # "electrochemical impedance",
    # "EIS",
    # Units/values
    "S/cm",
    "S cm",
    "siemens",
    # Material properties
    # "semiconductor",
    # "semiconducting",
    # "charge carrier",
    # "carrier mobility",
    # "electron mobility",
    # "hole mobility",
    # "band gap",
    # "bandgap",
    # "doping",
    # "doped",
    # "thermoelectric",
    # "solid electrolyte",
    # "ion conductor",
    # "proton conductor",
    # "superconductor",
    # "superconducting",
]

exclude_keywords = [
    "thermal conductivity",
    "photocatalysis",
]

# Apply filter
filtered_dataset = filter_dataset(
    dataset, text_column, list_keywords, exclude_keywords
)

print(f"Filtered: {len(filtered_dataset)} / {len(dataset)} papers")

In [ ]:
# Preview filtered results
filtered_dataset.to_pandas()[["id", "title", "pdf_url"]].head(100)

In [ ]:
# Check which keywords matched (optional - for analysis)
def count_keyword_matches(ds, text_column, keywords):
    counts = {}
    for kw in keywords:

        def has_keyword(example, keyword=kw):
            if example[text_column] is None:
                return False
            return keyword.lower() in example[text_column].lower()

        count = len(ds.filter(has_keyword))
        if count > 0:
            counts[kw] = count
    return dict(sorted(counts.items(), key=lambda x: -x[1]))


keyword_counts = count_keyword_matches(dataset, text_column, list_keywords)
print("Keyword match counts (before exclusion):")
for kw, count in keyword_counts.items():
    print(f"  {kw}: {count}")

In [ ]:
# Push to HuggingFace as a new split (creates a PR)
filtered_dataset.push_to_hub(
    repo_id="LeMaterial/LeMat-Synth-Papers",
    config_name="full",
    split="electrical_conductivity_keywords_only",  # New split name
    token=True,
    create_pr=True,
    commit_message="Add electrical_conductivity_keywords_only split filtered from thermocatalysis_keywords_and_LLM",
)

print("Created PR for new split 'electrical_conductivity_keywords_only'!")